In [1]:
!uv pip install datasets

Using Python 3.12.11 environment at: /home/zeus/miniconda3/envs/cloudspace
Audited 1 package in 75ms


In [2]:
import json
from datasets import load_dataset

# ========================
# 1. Load from Hugging Face
# ========================

print("Loading dataset from Hugging Face...")
dataset = load_dataset("newfacade/LeetCodeDataset", split="train")   # or "test" if you want test split

# Extract all question_id into a list
question_ids = list(dataset["question_id"])

print(f"Loaded {len(question_ids)} problems from Hugging Face.")
print(f"Sample question_ids: {question_ids[:10]}")

# ========================
# 2. Load your local explanations.json
# ========================

with open("explanations.json", "r", encoding="utf-8") as f:
    explanations_data = json.load(f)

print(f"Loaded {len(explanations_data)} explanations from local file.")

# ========================
# 3. Filter only English explanations
# ========================

filtered_explanations = [
    item for item in explanations_data 
    if item.get("is_english") is True
]

print(f"After filtering English only: {len(filtered_explanations)} problems remain.")

Loading dataset from Hugging Face...
Loaded 2641 problems from Hugging Face.
Sample question_ids: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
Loaded 3888 explanations from local file.
After filtering English only: 2773 problems remain.


In [3]:
import json
from datasets import load_dataset

# ========================
# 1. Load from Hugging Face
# ========================

print("Loading dataset from Hugging Face...")
dataset = load_dataset("newfacade/LeetCodeDataset", split="train")

# Store all question_ids in a list (as array)
hf_question_ids = list(dataset["question_id"])

print(f"Loaded {len(hf_question_ids)} problems from Hugging Face.")
print(f"Sample HF question_ids: {hf_question_ids[:10]}")

# Save HF question ids
with open("hf_question_ids.json", "w") as f:
    json.dump(hf_question_ids, f, indent=2)

# ========================
# 2. Load your local explanations.json
# ========================

with open("explanations.json", "r", encoding="utf-8") as f:
    explanations_data = json.load(f)

print(f"\nLoaded {len(explanations_data)} explanations from local file.")

# ========================
# 3. Filter only English explanations
# ========================

filtered_explanations = [
    item for item in explanations_data 
    if item.get("is_english") is True
]

# Extract English problem_ids from your explanations
english_problem_ids = [item["problem_id"] for item in filtered_explanations]

print(f"After filtering English only: {len(filtered_explanations)} problems remain.")

# ========================
# 4. Find common question_ids (Intersection)
# ========================

# Convert both to sets for fast intersection
hf_set = set(hf_question_ids)
english_set = set(english_problem_ids)

common_ids = sorted(hf_set.intersection(english_set))

print(f"\n📊 Summary:")
print(f"   Total in HF Dataset     : {len(hf_question_ids)}")
print(f"   English Explanations    : {len(english_problem_ids)}")
print(f"   Common Problems         : {len(common_ids)}")


Loading dataset from Hugging Face...


Loaded 2641 problems from Hugging Face.
Sample HF question_ids: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

Loaded 3888 explanations from local file.
After filtering English only: 2773 problems remain.

📊 Summary:
   Total in HF Dataset     : 2641
   English Explanations    : 2773
   Common Problems         : 2044


In [4]:
import json

# ========================
# 5. Build merged dataset
# ========================

# Create lookup dict for HF data
hf_lookup = {}
for item in dataset:
    hf_lookup[item['question_id']] = item

# Create lookup dict for local explanations (English only)
expl_lookup = {}
for item in filtered_explanations:
    expl_lookup[item['problem_id']] = item

# Merge for common IDs
merged_records = []
for pid in common_ids:
    hf_item = hf_lookup.get(pid)
    expl_item = expl_lookup.get(pid)
    
    if hf_item and expl_item:
        # Get explanation for solution 1, or empty string if missing
        explanation = ""
        if '1' in expl_item.get('explanations', {}):
            explanation = expl_item['explanations']['1'] or ""
        
        # Get time_complexity, or empty string if null
        time_complexity = expl_item.get('time_complexity', "") or ""
        
        record = {
            'problem_id': pid,
            'task_id': hf_item['task_id'],
            'difficulty': hf_item['difficulty'],
            'tags': hf_item['tags'],
            'problem_description': hf_item['problem_description'],
            'time_complexity': time_complexity,
            'explanation': explanation
        }
        merged_records.append(record)

print(f"Merged {len(merged_records)} records")

# ========================
# 6. Save to JSONL
# ========================

output_path = "dataset.jsonl"
with open(output_path, 'w', encoding='utf-8') as f:
    for record in merged_records:
        f.write(json.dumps(record, ensure_ascii=False) + '\n')

print(f"Saved to {output_path}")

# Preview
print("\nSample record:")
print(json.dumps(merged_records[0], indent=2, ensure_ascii=False)[:500] + "...")

Merged 2044 records
Saved to dataset.jsonl

Sample record:
{
  "problem_id": 1,
  "task_id": "two-sum",
  "difficulty": "Easy",
  "tags": [
    "Array",
    "Hash Table"
  ],
  "problem_description": "Given an array of integers nums and an integer target, return indices of the two numbers such that they add up to target.\nYou may assume that each input would have exactly one solution, and you may not use the same element twice.\nYou can return the answer in any order.\n \nExample 1:\n\nInput: nums = [2,7,11,15], target = 9\nOutput: [0,1]\nExplanation: B...
